# Personalized Gift Recommender — EDA, Modeling, and Prototype

This notebook performs: data loading, EDA, text preprocessing, TF-IDF content-based recommender, optional ALS (if user-item interactions exist), saving model artifacts, and writing a Streamlit prototype.

In [2]:
# Optional: install required packages (run in terminal if preferred)
#!pip install --quiet pandas numpy scikit-learn scipy joblib streamlit implicit faiss-cpu sentence-transformers

In [3]:
import os
import pandas as pd
import numpy as np
import time
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from scipy import sparse
print('Imports ok')

Imports ok


In [4]:
# Load dataset (CSV must be in the same folder as this notebook)
DATA_FILE = 'gift_dataset_50000.csv'
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f"{DATA_FILE} not found in the current folder. Place the CSV here and re-run the cell.")
df = pd.read_csv(DATA_FILE)
print('Loaded:', DATA_FILE)
print('Shape:', df.shape)
df.head()

Loaded: gift_dataset_50000.csv
Shape: (50000, 16)


,item_id,title,category,subcategory,price,currency,tags,description,brand,target_gender,age_group_label,age_min,age_max,rating,stock,created_at
0,100000,Classic Skincare Set,Beauty & Personal Care,Skincare Set,328.49,INR,"skincare_set,beauty_&_personal_care,portable,t...",Classic Skincare Set is crafted with care by F...,FitTrack,boys,teen,13,19,3.44,16,2025-01-03
1,100001,Minimal Notebooks,Stationery,Notebooks,118.72,INR,"notebooks,stationery,valentine,budget,office",Give the Minimal Notebooks as a thoughtful gif...,HandyGifts,boys,teen,13,19,4.16,414,2025-01-25
2,100002,Comfort Personalized Journal,Personalized Gifts,Personalized Journal,50.00,INR,"personalized_journal,personalized_gifts,portab...",Comfort Personalized Journal by FitTrack. A gr...,FitTrack,boys,young_adult,20,35,4.14,183,2025-06-20
3,100003,Cute Necklace,Jewelry,Necklace,861.88,INR,"necklace,jewelry,travel,for-her,valentine",Give the Cute Necklace as a thoughtful gift — ...,PeakActive,boys,adult,36,60,4.51,295,2025-04-23
4,100004,Elegant Travel Mug,Home & Kitchen,Travel Mug,50.00,INR,"travel_mug,home_&_kitchen,portable,anniversary...",Elegant Travel Mug is crafted with care by Cho...,ChocoDelight,unisex,teen,13,19,3.71,325,2025-06-06


In [5]:
# Basic EDA
print('\nColumns:')
print(list(df.columns))
print('\nNull counts (top 30):')
display(df.isnull().sum().sort_values(ascending=False).head(30))
print('\nData types:')
display(df.dtypes)


Columns:
['item_id', 'title', 'category', 'subcategory', 'price', 'currency', 'tags', 'description', 'brand', 'target_gender', 'age_group_label', 'age_min', 'age_max', 'rating', 'stock', 'created_at']

Null counts (top 30):


item_id            0
title              0
category           0
subcategory        0
price              0
currency           0
tags               0
description        0
brand              0
target_gender      0
age_group_label    0
age_min            0
age_max            0
rating             0
stock              0
created_at         0
dtype: int64


Data types:


item_id              int64
title               object
category            object
subcategory         object
price              float64
currency            object
tags                object
description         object
brand               object
target_gender       object
age_group_label     object
age_min              int64
age_max              int64
rating             float64
stock                int64
created_at          object
dtype: object

In [6]:
# Detect likely columns
cols = [c.lower() for c in df.columns]
def find_col(possible_names):
    for n in possible_names:
        if n in cols:
            return [c for c in df.columns if c.lower()==n][0]
    return None
product_id_col = find_col(['product_id','id','item_id','sku'])
title_col      = find_col(['title','name','product_name','item_name'])
desc_col       = find_col(['description','desc','details','product_description'])
price_col      = find_col(['price','cost','amount'])
category_col   = find_col(['category','categories','cat'])
image_col      = find_col(['image','image_url','img','picture','photo'])
views_col      = find_col(['views','view_count','impressions'])
purchases_col  = find_col(['purchases','purchase_count','orders','sales'])
clicks_col     = find_col(['clicks','click_count'])
timestamp_col  = find_col(['timestamp','time','event_time','created_at','date'])
print('Detected columns:')
print('product_id:', product_id_col)
print('title:', title_col)
print('description:', desc_col)
print('price:', price_col)
print('category:', category_col)
print('image:', image_col)
print('views:', views_col, 'purchases:', purchases_col, 'clicks:', clicks_col)
print('timestamp:', timestamp_col)

Detected columns:
product_id: item_id
title: title
description: description
price: price
category: category
image: None
views: None purchases: None clicks: None
timestamp: created_at


In [ ]:
# Basic checks
if product_id_col is None:
    df['_product_id_'] = df.index.astype(str)
    product_id_col = '_product_id_'
print('Unique products:', df[product_id_col].nunique())
if price_col:
    df['_price_num_'] = pd.to_numeric(df[price_col], errors='coerce')
    print('\nPrice stats:')
    display(df['_price_num_'].describe())
else:
    print('\nNo price column detected.')
if category_col:
    print('\nTop categories:')
    display(df[category_col].value_counts().head(10))

In [7]:
# Create combined text column
text_cols = []
if title_col: text_cols.append(title_col)
if desc_col: text_cols.append(desc_col)
for c in ['tags','category','features','attributes']:
    if c in df.columns and c not in text_cols:
        text_cols.append(c)
if not text_cols:
    text_cols = list(df.columns[:2])
print('Text columns used:', text_cols)
df['__text_for_rec__'] = df[text_cols].fillna('').astype(str).agg(' '.join, axis=1)
df[['__text_for_rec__']].head()

Text columns used: ['title', 'description', 'tags', 'category']


,__text_for_rec__
0,Classic Skincare Set Classic Skincare Set is c...
1,Minimal Notebooks Give the Minimal Notebooks a...
2,Comfort Personalized Journal Comfort Personali...
3,Cute Necklace Give the Cute Necklace as a thou...
4,Elegant Travel Mug Elegant Travel Mug is craft...


In [8]:
# Simple text cleaning
import re
def clean_text(s):
    s = str(s).lower()
    s = re.sub(r'http\S+',' ',s)
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s
df['__text_for_rec__'] = df['__text_for_rec__'].apply(clean_text)
df['__text_for_rec__'].head()

0    classic skincare set classic skincare set is c...
1    minimal notebooks give the minimal notebooks a...
2    comfort personalized journal comfort personali...
3    cute necklace give the cute necklace as a thou...
4    elegant travel mug elegant travel mug is craft...
Name: __text_for_rec__, dtype: object

In [9]:
# Build TF-IDF matrix
tfidf_max_features = 8000
tfidf = TfidfVectorizer(stop_words='english', max_features=tfidf_max_features)
t0 = time.time()
tfidf_matrix = tfidf.fit_transform(df['__text_for_rec__'].values)
print('TF-IDF matrix shape:', tfidf_matrix.shape, 'built in {:.1f}s'.format(time.time()-t0))

TF-IDF matrix shape: (50000, 188) built in 1.5s


In [10]:
# Recommend by free-text query
def recommend_by_text(query, topn=10):
    q = clean_text(query)
    q_vec = tfidf.transform([q])
    sims = linear_kernel(q_vec, tfidf_matrix).flatten()
    top_idx = sims.argsort()[-topn:][::-1]
    out = df.iloc[top_idx].copy()
    out['score'] = sims[top_idx]
    return out

example_query = 'birthday gift for a coffee lover'
res = recommend_by_text(example_query, topn=5)
display(res[[product_id_col, title_col, price_col, 'score']])

,item_id,title,price,score
47923,147923,Stylish Coffee Beans,2136.57,0.595126
31933,131933,Compact Coffee Beans,1075.73,0.594863
22549,122549,Eco Coffee Beans,1044.61,0.593211
39076,139076,Eco Coffee Beans,50.00,0.593196
31109,131109,Stylish Coffee Beans,143.77,0.585129


In [11]:
# Save TF-IDF artifacts
os.makedirs('models', exist_ok=True)
joblib.dump(tfidf, 'models/tfidf_vectorizer.joblib')
sparse.save_npz('models/tfidf_matrix.npz', tfidf_matrix)
cols_to_save = [c for c in [product_id_col, title_col, price_col, category_col, image_col, '__text_for_rec__'] if c]
df[cols_to_save].to_parquet('models/product_lookup.parquet', index=False)
print('Saved TF-IDF artifacts in ./models')

Saved TF-IDF artifacts in ./models


In [12]:
# Optional: prepare interactions if user-level data exists (no user column required in product CSV)
possible_user_cols = [c for c in df.columns if c.lower() in ('user_id','userid','user','buyer_id','customer_id')]
user_col = possible_user_cols[0] if possible_user_cols else None
if user_col:
    print('Detected user column for interactions:', user_col)
else:
    print('No user column detected. Skip ALS training unless an interactions file is available.')

No user column detected. Skip ALS training unless an interactions file is available.


In [13]:
# Write a simple Streamlit app scaffold that loads artifacts and serves recommendations
app_code = '''
import streamlit as st
import joblib
from scipy import sparse
import pandas as pd
from sklearn.metrics.pairwise import linear_kernel

@st.cache_data
def load_models():
    tfidf = joblib.load('models/tfidf_vectorizer.joblib')
    tfidf_mat = sparse.load_npz('models/tfidf_matrix.npz')
    products = pd.read_parquet('models/product_lookup.parquet')
    return tfidf, tfidf_mat, products

tfidf, tfidf_mat, products = load_models()
st.title('Gift Recommender')
query = st.text_input('Describe recipient + occasion + budget', 'birthday gift for a coffee lover')
k = st.slider('Number of results', 3, 20, 8)
if st.button('Recommend'):
    q_vec = tfidf.transform([query])
    sims = linear_kernel(q_vec, tfidf_mat).flatten()
    top_idx = sims.argsort()[-k:][::-1]
    res = products.iloc[top_idx].copy()
    res['score'] = sims[top_idx]
    for _, row in res.head(k).iterrows():
        st.write(f"**{row.get('title','')}** — {row.get('price','')}, score: {row['score']:.3f}")
'''
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)
print('Wrote app.py')

Wrote app.py


In [14]:
# Write requirements.txt
reqs = '''
pandas
numpy
scikit-learn
scipy
joblib
streamlit
implicit
sentence-transformers
faiss-cpu
'''
with open('requirements.txt','w') as f:
    f.write(reqs.strip())
print('Wrote requirements.txt')

Wrote requirements.txt


## Next steps
- If user-level interactions exist, prepare an interactions CSV (user_id, product_id, event, timestamp) and train ALS.
- Add budget and category filters in the Streamlit app.
- Containerize with Docker for deployment.